# LESSON 4.3: Basics of Filtering in the Frequency Domain
## Filtering in the Frequency Domain

In this lesson:
- The frequency domain filtering equation
- Zero-padding to avoid wraparound error
- The step-by-step frequency domain filtering procedure
- Filtering with and without padding (wraparound error demonstration)
- Zero-phase-shift filters
- Relationship between spatial kernels and frequency domain filters
- Complete example: lowpass filtering step by step

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. The Frequency Domain Filtering Equation

The fundamental equation of frequency domain filtering is:

$$\boxed{g(x, y) = \mathfrak{F}^{-1}\left[H(u, v) \cdot F(u, v)\right]}$$

Where:
- $f(x, y)$ = the input image in the **spatial domain**
- $F(u, v) = \mathfrak{F}\{f(x, y)\}$ = the **DFT** of the input image (complex-valued)
- $H(u, v)$ = the **filter transfer function** defined in the frequency domain
- $H(u, v) \cdot F(u, v)$ = element-wise multiplication in the frequency domain
- $\mathfrak{F}^{-1}$ = the **inverse DFT**
- $g(x, y)$ = the **filtered output image** in the spatial domain

### Connection to spatial filtering (Convolution Theorem)

By the convolution theorem:

$$g(x, y) = f(x, y) \star h(x, y) \quad \Leftrightarrow \quad G(u, v) = H(u, v) \cdot F(u, v)$$

Where $h(x, y)$ is the spatial domain kernel corresponding to $H(u, v)$.

**Key insight**: Convolution in the spatial domain (expensive for large kernels) becomes simple element-wise multiplication in the frequency domain.

In [ ]:
# Demonstrate the basic filtering equation
# Create a test image: white rectangle on black background
M, N = 256, 256
f = np.zeros((M, N), dtype=np.float64)
f[78:178, 78:178] = 255  # 100x100 white square

# Compute DFT of the image
F = np.fft.fft2(f)
F_centered = np.fft.fftshift(F)

# Create a simple lowpass filter H(u,v): ideal circular lowpass
u = np.arange(M) - M // 2
v = np.arange(N) - N // 2
U, V = np.meshgrid(v, u)
D = np.sqrt(U**2 + V**2)  # distance from center
D0 = 30  # cutoff frequency
H = (D <= D0).astype(np.float64)  # ideal lowpass

# Apply the filtering equation: G = H * F
G_centered = H * F_centered

# Inverse DFT to get filtered image
G = np.fft.ifftshift(G_centered)
g = np.real(np.fft.ifft2(G))

# Visualize each step
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

axes[0, 0].imshow(f, cmap='gray')
axes[0, 0].set_title('$f(x,y)$\nInput Image', fontsize=12)
axes[0, 0].axis('off')

axes[0, 1].imshow(np.log1p(np.abs(F_centered)), cmap='gray')
axes[0, 1].set_title('$|F(u,v)|$\nSpectrum of Input', fontsize=12)
axes[0, 1].axis('off')

axes[0, 2].imshow(H, cmap='gray')
axes[0, 2].set_title('$H(u,v)$\nFilter (Ideal Lowpass)', fontsize=12)
axes[0, 2].axis('off')

axes[1, 0].imshow(np.log1p(np.abs(G_centered)), cmap='gray')
axes[1, 0].set_title('$|H(u,v) \\cdot F(u,v)|$\nFiltered Spectrum', fontsize=12)
axes[1, 0].axis('off')

axes[1, 1].imshow(g, cmap='gray')
axes[1, 1].set_title('$g(x,y) = \\mathfrak{F}^{-1}[H \\cdot F]$\nFiltered Output', fontsize=12)
axes[1, 1].axis('off')

# Show before/after comparison
axes[1, 2].imshow(np.hstack([f, g]), cmap='gray')
axes[1, 2].set_title('Before (left) vs After (right)', fontsize=12)
axes[1, 2].axis('off')

plt.suptitle('Frequency Domain Filtering: $g(x,y) = \\mathfrak{F}^{-1}[H(u,v) \\cdot F(u,v)]$',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("The lowpass filter removes high frequencies (edges become blurred).")

## 2. Zero-Padding to Avoid Wraparound Error

### The Problem: Circular Convolution

The DFT computes **circular (cyclic) convolution**, not the linear convolution we want. This means the convolution "wraps around" the borders, causing artifacts.

### Why does this happen?

The DFT treats the image as **periodic** -- it assumes the signal repeats infinitely in all directions. When we multiply in the frequency domain, the resulting spatial convolution wraps around the image boundaries.

### The Solution: Zero-Padding

If the image $f(x,y)$ has size $A \times B$ and the filter kernel $h(x,y)$ has size $C \times D$, then we must pad both to size $P \times Q$ where:

$$\boxed{P \geq A + C - 1, \quad Q \geq B + D - 1}$$

This ensures the circular convolution result matches the linear convolution result (no wraparound artifacts).

### Padding procedure:
1. Place $f(x,y)$ in the top-left corner of a $P \times Q$ array of zeros
2. Place $h(x,y)$ in the top-left corner of another $P \times Q$ array of zeros
3. Compute DFTs, multiply, take IDFT
4. Crop the result back to the original image size $A \times B$

In [ ]:
# Visual demonstration of why zero-padding is needed
# 1-D example first for clarity

# Signal f and kernel h
f_1d = np.array([0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0], dtype=np.float64)
h_1d = np.array([0.2, 0.2, 0.2, 0.2, 0.2], dtype=np.float64)  # averaging kernel

A = len(f_1d)  # 12
C = len(h_1d)  # 5
P = A + C - 1  # 16 (required padded size)

# Method 1: DFT multiplication WITHOUT padding (causes wraparound)
h_same_size = np.zeros(A)
h_same_size[:C] = h_1d
F_no_pad = np.fft.fft(f_1d)
H_no_pad = np.fft.fft(h_same_size)
result_no_pad = np.real(np.fft.ifft(F_no_pad * H_no_pad))

# Method 2: DFT multiplication WITH proper padding
f_padded = np.zeros(P)
f_padded[:A] = f_1d
h_padded = np.zeros(P)
h_padded[:C] = h_1d
F_padded = np.fft.fft(f_padded)
H_padded_dft = np.fft.fft(h_padded)
result_padded = np.real(np.fft.ifft(F_padded * H_padded_dft))

# Method 3: True linear convolution (ground truth)
result_linear = np.convolve(f_1d, h_1d, mode='full')

fig, axes = plt.subplots(4, 1, figsize=(14, 10))

axes[0].stem(np.arange(A), f_1d, linefmt='b-', markerfmt='bo', basefmt='k-')
axes[0].set_title(f'Signal f (length A={A}) and kernel h (length C={C})', fontsize=12)
axes[0].set_ylabel('f(x)')
axes[0].grid(True, alpha=0.3)
axes[0].set_xlim(-1, P+1)

axes[1].stem(np.arange(A), result_no_pad, linefmt='r-', markerfmt='ro', basefmt='k-')
axes[1].set_title('DFT multiply WITHOUT padding (CIRCULAR convolution - wraparound error!)', fontsize=12)
axes[1].set_ylabel('g(x)')
axes[1].grid(True, alpha=0.3)
axes[1].set_xlim(-1, P+1)

axes[2].stem(np.arange(P), result_padded, linefmt='g-', markerfmt='go', basefmt='k-')
axes[2].set_title(f'DFT multiply WITH padding (P={P} >= A+C-1={A+C-1}) - CORRECT', fontsize=12)
axes[2].set_ylabel('g(x)')
axes[2].grid(True, alpha=0.3)
axes[2].set_xlim(-1, P+1)

axes[3].stem(np.arange(len(result_linear)), result_linear, linefmt='m-', markerfmt='m^', basefmt='k-')
axes[3].set_title('True linear convolution (ground truth)', fontsize=12)
axes[3].set_ylabel('g(x)')
axes[3].grid(True, alpha=0.3)
axes[3].set_xlim(-1, P+1)

plt.suptitle('Zero-Padding: Circular vs Linear Convolution (1-D)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Signal length A = {A}, Kernel length C = {C}")
print(f"Required padded size P >= A + C - 1 = {A} + {C} - 1 = {A + C - 1}")
print(f"Max error without padding: {np.max(np.abs(result_no_pad - result_padded[:A])):.6f}")
print(f"Max error with padding vs linear: {np.max(np.abs(result_padded - result_linear)):.2e}")

In [ ]:
# 2-D zero-padding demonstration
# Create a small image and kernel to visualize

# Image f(x,y): A x B = 64 x 64
A, B = 64, 64
f_small = np.zeros((A, B), dtype=np.float64)
f_small[16:48, 16:48] = 255  # white square

# Kernel h(x,y): C x D = 21 x 21 (averaging filter)
C, D = 21, 21
h_kernel = np.ones((C, D), dtype=np.float64) / (C * D)

# Required padded size
P = A + C - 1  # 84
Q = B + D - 1  # 84

# Show the padding concept visually
f_padded_2d = np.zeros((P, Q), dtype=np.float64)
f_padded_2d[:A, :B] = f_small

h_padded_2d = np.zeros((P, Q), dtype=np.float64)
h_padded_2d[:C, :D] = h_kernel

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(f_small, cmap='gray', extent=[0, B, A, 0])
axes[0].set_title(f'Original Image\n$A \\times B = {A} \\times {B}$', fontsize=12)
axes[0].set_xlabel('columns')
axes[0].set_ylabel('rows')

axes[1].imshow(f_padded_2d, cmap='gray', extent=[0, Q, P, 0])
axes[1].axhline(y=A, color='r', linestyle='--', linewidth=2)
axes[1].axvline(x=B, color='r', linestyle='--', linewidth=2)
axes[1].set_title(f'Zero-Padded Image\n$P \\times Q = {P} \\times {Q}$', fontsize=12)
axes[1].set_xlabel('columns')
axes[1].set_ylabel('rows')

# Show padded kernel (scaled for visibility)
h_vis = h_padded_2d.copy()
h_vis = h_vis / h_vis.max()  # normalize to [0,1]
axes[2].imshow(h_vis, cmap='gray', extent=[0, Q, P, 0])
axes[2].set_title(f'Zero-Padded Kernel\n$C \\times D = {C} \\times {D}$ in $P \\times Q$ array', fontsize=12)
axes[2].set_xlabel('columns')
axes[2].set_ylabel('rows')

plt.suptitle(f'Zero-Padding: P >= A+C-1 = {A}+{C}-1 = {P}, Q >= B+D-1 = {B}+{D}-1 = {Q}',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Step-by-Step Frequency Domain Filtering Procedure

The complete procedure for filtering an image of size $M \times N$ with a filter kernel of size $m \times n$ consists of **8 steps** (Gonzalez, Section 4.7):

### The 8 Steps:

1. **Determine padding size**: Given $f(x,y)$ of size $M \times N$ and filter $h(x,y)$ of size $m \times n$, compute $P \geq M + m - 1$ and $Q \geq N + n - 1$.

2. **Zero-pad the image**: Form the padded image $f_p(x,y)$ of size $P \times Q$ by appending zeros to $f(x,y)$.

3. **Center the transform**: Multiply $f_p(x,y)$ by $(-1)^{x+y}$ to center the DFT.

4. **Compute the DFT**: Compute $F(u,v) = \text{DFT}\{f_p(x,y) \cdot (-1)^{x+y}\}$.

5. **Construct the filter**: Generate the filter function $H(u,v)$ of size $P \times Q$ with center at $(P/2, Q/2)$.

6. **Apply the filter**: Compute $G(u,v) = H(u,v) \cdot F(u,v)$.

7. **Compute inverse DFT**: Compute $g_p(x,y) = \text{real}\left\{\text{IDFT}\{G(u,v)\}\right\} \cdot (-1)^{x+y}$.

8. **Crop the result**: Extract the $M \times N$ region from the top-left corner of $g_p(x,y)$ to obtain the final result $g(x,y)$.

**Note**: When the filter $H(u,v)$ is specified directly in the frequency domain (rather than derived from a spatial kernel), we typically skip step 1 (using $P = 2M$, $Q = 2N$ as a safe choice) and construct $H(u,v)$ directly in step 5.

In [ ]:
# Implement the 8-step frequency domain filtering procedure

# Create a test image with distinct features
M, N = 128, 128
f = np.zeros((M, N), dtype=np.float64)
# Add a rectangle
f[30:100, 30:100] = 200
# Add a smaller bright rectangle
f[50:80, 50:80] = 255
# Add some thin lines (high frequency content)
f[20, 20:110] = 255
f[20:110, 20] = 255

# Define a spatial averaging kernel
kernel_size = 15
h = np.ones((kernel_size, kernel_size), dtype=np.float64) / (kernel_size**2)
m, n = h.shape

print("="*60)
print("STEP-BY-STEP FREQUENCY DOMAIN FILTERING")
print("="*60)

# STEP 1: Determine padding size
P = M + m - 1
Q = N + n - 1
print(f"\nStep 1: Padding size")
print(f"  Image size: M={M}, N={N}")
print(f"  Kernel size: m={m}, n={n}")
print(f"  Padded size: P={P} (>= {M}+{m}-1={M+m-1}), Q={Q} (>= {N}+{n}-1={N+n-1})")

# STEP 2: Zero-pad the image
fp = np.zeros((P, Q), dtype=np.float64)
fp[:M, :N] = f
print(f"\nStep 2: Image zero-padded to {P}x{Q}")

# STEP 3: Multiply by (-1)^(x+y) to center
x_idx = np.arange(P)
y_idx = np.arange(Q)
X, Y = np.meshgrid(y_idx, x_idx)
centering = (-1.0) ** (X + Y)
fp_centered = fp * centering
print(f"\nStep 3: Multiplied by (-1)^(x+y) to center the DFT")

# STEP 4: Compute the DFT
Fp = np.fft.fft2(fp_centered)
print(f"\nStep 4: Computed DFT of padded, centered image")

# STEP 5: Construct the filter H(u,v) of size P x Q
# Pad the kernel to P x Q and compute its DFT
hp = np.zeros((P, Q), dtype=np.float64)
hp[:m, :n] = h
hp_centered = hp * centering
Hp = np.fft.fft2(hp_centered)
print(f"\nStep 5: Constructed filter H(u,v) of size {P}x{Q}")

# STEP 6: Apply the filter
Gp = Hp * Fp
print(f"\nStep 6: Applied filter: G(u,v) = H(u,v) * F(u,v)")

# STEP 7: Compute inverse DFT and uncenter
gp = np.real(np.fft.ifft2(Gp)) * centering
print(f"\nStep 7: Computed IDFT and multiplied by (-1)^(x+y)")

# STEP 8: Crop to original size
g = gp[:M, :N]
print(f"\nStep 8: Cropped result to {M}x{N}")

# Verify against scipy convolve
from scipy.signal import fftconvolve
g_reference = fftconvolve(f, h, mode='full')[:M, :N]
print(f"\nVerification: Max error vs scipy fftconvolve = {np.max(np.abs(g - g_reference)):.2e}")

In [ ]:
# Visualize each step
fig, axes = plt.subplots(2, 4, figsize=(18, 9))

axes[0, 0].imshow(f, cmap='gray')
axes[0, 0].set_title('Step 0: Original $f(x,y)$\n$128 \\times 128$', fontsize=11)
axes[0, 0].axis('off')

axes[0, 1].imshow(fp, cmap='gray')
axes[0, 1].set_title(f'Step 2: Zero-padded $f_p$\n${P} \\times {Q}$', fontsize=11)
axes[0, 1].axis('off')

axes[0, 2].imshow(fp_centered, cmap='gray')
axes[0, 2].set_title('Step 3: $f_p \\cdot (-1)^{x+y}$\n(centered)', fontsize=11)
axes[0, 2].axis('off')

axes[0, 3].imshow(np.log1p(np.abs(Fp)), cmap='gray')
axes[0, 3].set_title('Step 4: $|F(u,v)|$\n(centered DFT)', fontsize=11)
axes[0, 3].axis('off')

axes[1, 0].imshow(np.log1p(np.abs(Hp)), cmap='gray')
axes[1, 0].set_title('Step 5: $|H(u,v)|$\n(filter in freq domain)', fontsize=11)
axes[1, 0].axis('off')

axes[1, 1].imshow(np.log1p(np.abs(Gp)), cmap='gray')
axes[1, 1].set_title('Step 6: $|G(u,v)| = |H \\cdot F|$', fontsize=11)
axes[1, 1].axis('off')

axes[1, 2].imshow(gp, cmap='gray')
axes[1, 2].set_title(f'Step 7: IDFT result $g_p$\n${P} \\times {Q}$', fontsize=11)
axes[1, 2].axis('off')

axes[1, 3].imshow(g, cmap='gray')
axes[1, 3].set_title(f'Step 8: Cropped $g(x,y)$\n${M} \\times {N}$', fontsize=11)
axes[1, 3].axis('off')

plt.suptitle('The 8-Step Frequency Domain Filtering Procedure', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Filtering With and Without Padding: Wraparound Error

Now let us demonstrate the **wraparound error** that occurs when filtering is performed without proper zero-padding.

Without padding, the DFT computes a **circular convolution**, where the signal wraps around at the boundaries. This creates artifacts particularly visible near the edges of the image.

In [ ]:
# Create a test image where wraparound is clearly visible
M, N = 128, 128
f = np.zeros((M, N), dtype=np.float64)

# Place bright features near the edges so wraparound is obvious
f[5:25, 5:125] = 255     # top bar
f[105:125, 5:125] = 255  # bottom bar
f[5:125, 5:25] = 255     # left bar
f[5:125, 105:125] = 255  # right bar

# Define a large averaging kernel (makes wraparound more visible)
k = 21
h = np.ones((k, k), dtype=np.float64) / (k * k)

# --- Method 1: WITHOUT padding (wraparound error) ---
F_nopad = np.fft.fft2(f)
# Pad kernel to image size and compute DFT
h_nopad = np.zeros((M, N), dtype=np.float64)
h_nopad[:k, :k] = h
H_nopad = np.fft.fft2(h_nopad)
G_nopad = H_nopad * F_nopad
g_nopad = np.real(np.fft.ifft2(G_nopad))

# --- Method 2: WITH proper padding ---
P = M + k - 1
Q = N + k - 1
f_padded = np.zeros((P, Q), dtype=np.float64)
f_padded[:M, :N] = f
h_padded = np.zeros((P, Q), dtype=np.float64)
h_padded[:k, :k] = h
F_padded = np.fft.fft2(f_padded)
H_padded = np.fft.fft2(h_padded)
G_padded = H_padded * F_padded
g_padded = np.real(np.fft.ifft2(G_padded))[:M, :N]  # crop

# --- Method 3: Spatial convolution (ground truth) ---
from scipy.signal import fftconvolve
g_truth = fftconvolve(f, h, mode='full')[:M, :N]

# Compute error maps
error_nopad = np.abs(g_nopad - g_truth)
error_padded = np.abs(g_padded - g_truth)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

axes[0, 0].imshow(f, cmap='gray')
axes[0, 0].set_title('Original Image\n(features near edges)', fontsize=12)
axes[0, 0].axis('off')

axes[0, 1].imshow(g_nopad, cmap='gray')
axes[0, 1].set_title('WITHOUT Padding\n(WRAPAROUND ERROR)', fontsize=12)
axes[0, 1].axis('off')

axes[0, 2].imshow(g_padded, cmap='gray')
axes[0, 2].set_title('WITH Padding\n(CORRECT result)', fontsize=12)
axes[0, 2].axis('off')

axes[1, 0].imshow(g_truth, cmap='gray')
axes[1, 0].set_title('Ground Truth\n(scipy fftconvolve)', fontsize=12)
axes[1, 0].axis('off')

im1 = axes[1, 1].imshow(error_nopad, cmap='hot')
axes[1, 1].set_title(f'Error WITHOUT padding\nMax = {error_nopad.max():.2f}', fontsize=12)
axes[1, 1].axis('off')
plt.colorbar(im1, ax=axes[1, 1], fraction=0.046)

im2 = axes[1, 2].imshow(error_padded, cmap='hot')
axes[1, 2].set_title(f'Error WITH padding\nMax = {error_padded.max():.2e}', fontsize=12)
axes[1, 2].axis('off')
plt.colorbar(im2, ax=axes[1, 2], fraction=0.046)

plt.suptitle('Wraparound Error: Without vs With Zero-Padding',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Image size: {M}x{N}, Kernel size: {k}x{k}")
print(f"Padded size: {P}x{Q}")
print(f"Maximum error WITHOUT padding: {error_nopad.max():.4f}")
print(f"Maximum error WITH padding:    {error_padded.max():.2e}")
print("\nWraparound artifacts are clearly visible near the edges!")

In [ ]:
# Zoomed comparison to highlight wraparound artifacts
# Focus on the top-left corner where wraparound is most visible

region = 40  # pixels to show

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(f[:region, :region], cmap='gray', interpolation='nearest')
axes[0].set_title('Original (top-left corner)', fontsize=12)

axes[1].imshow(g_nopad[:region, :region], cmap='gray', interpolation='nearest')
axes[1].set_title('No padding (wraparound visible)', fontsize=12)

axes[2].imshow(g_padded[:region, :region], cmap='gray', interpolation='nearest')
axes[2].set_title('With padding (correct)', fontsize=12)

plt.suptitle('Zoomed View: Top-Left Corner\nWraparound causes energy from bottom/right edges to leak into top-left',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Zero-Phase-Shift Filters

### The Phase-Shift Problem

A general filter $H(u,v)$ is complex-valued:

$$H(u,v) = |H(u,v)| \, e^{j\phi_H(u,v)}$$

The phase $\phi_H(u,v)$ of the filter **shifts** the spatial position of features in the output. This is generally undesirable in image processing -- we want to change amplitudes without moving things around.

### Solution: Real, Symmetric Filters

If $H(u,v)$ is **real** and **symmetric** about the origin (i.e., $H(u,v) = H(-u,-v)$), then:

$$\phi_H(u,v) = 0 \quad \text{(zero phase shift)}$$

This means the filter only modifies the **magnitude** of $F(u,v)$ without altering the **phase** (which carries spatial/structural information).

### Why is this important?

- Recall from Lesson 4.2 that the **phase** carries most of the structural information
- A zero-phase-shift filter preserves all spatial relationships
- All standard frequency domain filters (lowpass, highpass, bandpass) are designed to be **real and symmetric**

### Mathematical proof

If $H(u,v)$ is real and symmetric:
- $\text{Im}\{H(u,v)\} = 0$, so $\phi_H = \arctan(0/R) = 0$
- $G(u,v) = H(u,v) \cdot F(u,v) = |H(u,v)| \cdot |F(u,v)| \cdot e^{j\phi_F(u,v)}$
- The phase of $G$ equals the phase of $F$ -- no shift is introduced!

In [ ]:
# Demonstrate zero-phase-shift vs phase-shifting filter

M, N = 256, 256
# Create a test image with clear spatial features
f = np.zeros((M, N), dtype=np.float64)
# Cross pattern
f[118:138, 50:206] = 200   # horizontal bar
f[50:206, 118:138] = 200   # vertical bar
# Small bright dot at center
Y, X = np.ogrid[-128:128, -128:128]
f[X**2 + Y**2 <= 10**2] = 255

# Compute centered DFT
F = np.fft.fftshift(np.fft.fft2(f))

# --- Filter 1: Real, symmetric (zero-phase-shift) ---
u = np.arange(M) - M // 2
v = np.arange(N) - N // 2
U, V = np.meshgrid(v, u)
D = np.sqrt(U**2 + V**2)
D0 = 40
H_real = np.exp(-(D**2) / (2 * D0**2))  # Gaussian lowpass (real, symmetric)

G1 = H_real * F
g1 = np.real(np.fft.ifft2(np.fft.ifftshift(G1)))

# --- Filter 2: Complex filter with phase shift ---
# Add a linear phase (corresponds to spatial shift)
shift_x, shift_y = 20, 15
phase_shift = np.exp(-1j * 2 * np.pi * (shift_x * U / M + shift_y * V / N))
H_complex = H_real * phase_shift  # same magnitude, but with phase

G2 = H_complex * F
g2 = np.real(np.fft.ifft2(np.fft.ifftshift(G2)))

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

axes[0, 0].imshow(f, cmap='gray')
axes[0, 0].set_title('Original Image', fontsize=12)
axes[0, 0].axis('off')

axes[0, 1].imshow(H_real, cmap='gray')
axes[0, 1].set_title('$H_1(u,v)$: Real & Symmetric\n(Zero phase shift)', fontsize=12)
axes[0, 1].axis('off')

axes[0, 2].imshow(g1, cmap='gray')
axes[0, 2].set_title('Filtered with $H_1$\n(blurred but NOT shifted)', fontsize=12)
axes[0, 2].axis('off')

axes[1, 0].imshow(np.abs(H_complex), cmap='gray')
axes[1, 0].set_title('$|H_2(u,v)|$: Same magnitude', fontsize=12)
axes[1, 0].axis('off')

axes[1, 1].imshow(np.angle(H_complex), cmap='hsv')
axes[1, 1].set_title('$\\angle H_2(u,v)$: Non-zero phase!', fontsize=12)
axes[1, 1].axis('off')

axes[1, 2].imshow(g2, cmap='gray')
axes[1, 2].set_title('Filtered with $H_2$\n(blurred AND SHIFTED!)', fontsize=12)
axes[1, 2].axis('off')

plt.suptitle('Zero-Phase-Shift Filter vs Complex Filter\n'
             'Real, symmetric H(u,v) preserves spatial positions',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Filter H1 (real, symmetric): Output is blurred but features stay in place.")
print("Filter H2 (complex, with phase): Output is blurred AND shifted!")
print(f"\nH1 is real? Max imaginary part: {np.max(np.abs(np.imag(H_real))):.2e}")
print(f"H2 is real? Max imaginary part: {np.max(np.abs(np.imag(H_complex))):.4f}")

## 6. Relationship Between Spatial Kernels and Frequency Domain Filters

The spatial domain kernel $h(x,y)$ and the frequency domain filter $H(u,v)$ are a **Fourier transform pair**:

$$h(x,y) \xleftrightarrow{\mathfrak{F}} H(u,v)$$

This means:
- $H(u,v) = \mathfrak{F}\{h(x,y)\}$ (DFT of the spatial kernel)
- $h(x,y) = \mathfrak{F}^{-1}\{H(u,v)\}$ (IDFT of the frequency filter)

### Key relationships:

| Spatial domain $h(x,y)$ | Frequency domain $H(u,v)$ |
|-------------------------|---------------------------|
| Small kernel (few pixels) | Broad frequency response |
| Large kernel (many pixels) | Narrow frequency response |
| Averaging (box) filter | Sinc-like function |
| Gaussian kernel | Gaussian (also Gaussian!) |
| Laplacian kernel | Emphasizes high frequencies |

### The Gaussian is special!

The Fourier transform of a Gaussian is also a Gaussian:

$$h(x,y) = e^{-\frac{x^2+y^2}{2\sigma^2}} \quad \Leftrightarrow \quad H(u,v) = 2\pi\sigma^2 \, e^{-2\pi^2\sigma^2(u^2+v^2)}$$

A **wider** Gaussian in space ($\sigma$ large) corresponds to a **narrower** Gaussian in frequency (stronger lowpass effect), and vice versa.

In [ ]:
# Show the frequency response of common spatial kernels

size = 128

# Kernel 1: Box/Averaging filter (different sizes)
def make_kernel_image(kernel, size):
    """Place kernel in center of a size x size image."""
    img = np.zeros((size, size), dtype=np.float64)
    kh, kw = kernel.shape
    r0 = size // 2 - kh // 2
    c0 = size // 2 - kw // 2
    img[r0:r0+kh, c0:c0+kw] = kernel
    return img

# Box filters of different sizes
box3 = np.ones((3, 3)) / 9.0
box9 = np.ones((9, 9)) / 81.0
box21 = np.ones((21, 21)) / 441.0

# Gaussian filters of different sigmas
def gaussian_kernel(size_k, sigma):
    ax = np.arange(-size_k // 2 + 1, size_k // 2 + 1)
    xx, yy = np.meshgrid(ax, ax)
    kernel = np.exp(-(xx**2 + yy**2) / (2 * sigma**2))
    return kernel / kernel.sum()

gauss_s = gaussian_kernel(11, 1.0)
gauss_m = gaussian_kernel(21, 3.0)
gauss_l = gaussian_kernel(41, 7.0)

# Laplacian kernel
laplacian = np.array([[0, 1, 0],
                      [1, -4, 1],
                      [0, 1, 0]], dtype=np.float64)

kernels = [box3, box9, box21, gauss_s, gauss_m, gauss_l, laplacian]
names = ['Box 3x3', 'Box 9x9', 'Box 21x21',
         'Gaussian $\\sigma=1$', 'Gaussian $\\sigma=3$', 'Gaussian $\\sigma=7$',
         'Laplacian']

fig, axes = plt.subplots(2, len(kernels), figsize=(22, 7))

for i, (kern, name) in enumerate(zip(kernels, names)):
    # Place kernel in center
    k_img = make_kernel_image(kern, size)
    
    # Compute frequency response
    H = np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(k_img)))
    H_mag = np.abs(H)
    
    # Show kernel (zoomed)
    kh, kw = kern.shape
    axes[0, i].imshow(kern, cmap='gray', interpolation='nearest')
    axes[0, i].set_title(f'{name}\n({kh}x{kw})', fontsize=10)
    axes[0, i].axis('off')
    
    # Show frequency response
    axes[1, i].imshow(H_mag, cmap='gray')
    axes[1, i].set_title('$|H(u,v)|$', fontsize=10)
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Spatial kernel\n$h(x,y)$', fontsize=12)
axes[1, 0].set_ylabel('Frequency response\n$|H(u,v)|$', fontsize=12)

plt.suptitle('Spatial Kernels and Their Frequency Responses: $h(x,y) \\leftrightarrow H(u,v)$',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Demonstrate the inverse relationship between spatial and frequency width
# Wider Gaussian in space -> Narrower Gaussian in frequency (stronger lowpass)

size = 256
sigmas = [1, 3, 7, 15]

fig, axes = plt.subplots(2, len(sigmas), figsize=(18, 8))

for i, sigma in enumerate(sigmas):
    # Create Gaussian kernel
    k_size = min(6 * sigma + 1, size - 1)
    if k_size % 2 == 0:
        k_size += 1
    kern = gaussian_kernel(int(k_size), sigma)
    k_img = make_kernel_image(kern, size)
    
    # Frequency response
    H = np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(k_img)))
    H_mag = np.abs(H)
    
    # 1-D cross-section through center
    spatial_profile = k_img[size // 2, :]
    freq_profile = H_mag[size // 2, :]
    
    axes[0, i].plot(spatial_profile, 'b-', linewidth=2)
    axes[0, i].set_title(f'$h(x)$: $\\sigma = {sigma}$', fontsize=12)
    axes[0, i].set_xlim([size//2 - 50, size//2 + 50])
    axes[0, i].grid(True, alpha=0.3)
    if i == 0:
        axes[0, i].set_ylabel('Spatial kernel', fontsize=12)
    
    axes[1, i].plot(freq_profile, 'r-', linewidth=2)
    axes[1, i].set_title(f'$|H(u)|$: narrower!', fontsize=12)
    axes[1, i].set_xlim([size//2 - 50, size//2 + 50])
    axes[1, i].set_ylim([0, 1.1])
    axes[1, i].grid(True, alpha=0.3)
    if i == 0:
        axes[1, i].set_ylabel('Frequency response', fontsize=12)

plt.suptitle('Inverse Relationship: Wider Gaussian in Space = Narrower in Frequency\n'
             '(Larger $\\sigma$ = Stronger lowpass filtering)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Complete Example: Lowpass Filtering Step by Step

Let us now apply the complete frequency domain filtering procedure to a synthetic biomedical-style image using a **Gaussian lowpass filter**.

### Gaussian Lowpass Filter

$$H(u,v) = e^{-D^2(u,v) / 2D_0^2}$$

Where:
- $D(u,v) = \sqrt{(u - P/2)^2 + (v - Q/2)^2}$ is the distance from the center
- $D_0$ is the **cutoff frequency** (controls the amount of smoothing)

Properties:
- $H(0,0) = 1$ (DC component passes through)
- $H(u,v) \to 0$ as $D \to \infty$ (high frequencies are attenuated)
- Smooth transition (no ringing artifacts unlike ideal lowpass)
- Real and symmetric (zero-phase-shift)

In [ ]:
# Create a synthetic biomedical-style test image
M, N = 256, 256
f = np.zeros((M, N), dtype=np.float64)

# Background tissue (slight noise)
np.random.seed(42)
f += np.random.normal(40, 5, (M, N))

# Large circular structure (simulating an organ cross-section)
Y, X = np.ogrid[-128:128, -128:128]
f[X**2 + Y**2 <= 90**2] += 60

# Smaller circular structure inside (simulating a lesion)
f[(X-20)**2 + (Y+15)**2 <= 25**2] += 80

# Another small structure
f[(X+30)**2 + (Y-25)**2 <= 15**2] += 100

# Add some fine detail / noise (high frequency)
f += np.random.normal(0, 10, (M, N))

# Clip to valid range
f = np.clip(f, 0, 255)

plt.figure(figsize=(6, 6))
plt.imshow(f, cmap='gray')
plt.title('Synthetic Biomedical Image\n(organ cross-section with lesions and noise)', fontsize=12)
plt.colorbar(label='Intensity')
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# ===== COMPLETE STEP-BY-STEP LOWPASS FILTERING =====

print("=" * 65)
print("COMPLETE FREQUENCY DOMAIN LOWPASS FILTERING PROCEDURE")
print("=" * 65)

# STEP 1: Determine padding size
# For filters defined directly in frequency domain, use P=2M, Q=2N
P, Q = 2 * M, 2 * N
print(f"\nStep 1: Padded size P={P}, Q={Q} (2M x 2N for frequency domain filters)")

# STEP 2: Zero-pad the image
fp = np.zeros((P, Q), dtype=np.float64)
fp[:M, :N] = f
print(f"Step 2: Image zero-padded from {M}x{N} to {P}x{Q}")

# STEP 3: Multiply by (-1)^(x+y)
x_idx = np.arange(P)
y_idx = np.arange(Q)
XG, YG = np.meshgrid(y_idx, x_idx)
centering = (-1.0) ** (XG + YG)
fp_centered = fp * centering
print(f"Step 3: Multiplied by (-1)^(x+y) to center the transform")

# STEP 4: Compute the DFT
Fp = np.fft.fft2(fp_centered)
print(f"Step 4: Computed 2-D DFT")

# STEP 5: Create the Gaussian lowpass filter H(u,v)
D0 = 30  # cutoff frequency
u = np.arange(P) - P // 2
v = np.arange(Q) - Q // 2
U, V = np.meshgrid(v, u)
D = np.sqrt(U**2 + V**2)
H = np.exp(-(D**2) / (2 * D0**2))
print(f"Step 5: Created Gaussian lowpass filter H(u,v) with D0={D0}")
print(f"         H is real: {np.all(np.isreal(H))}")
print(f"         H is symmetric: {np.allclose(H, np.flip(H))}")
print(f"         H(center) = {H[P//2, Q//2]:.4f}")

# STEP 6: Apply the filter
Gp = H * Fp
print(f"Step 6: Applied filter G(u,v) = H(u,v) * F(u,v)")

# STEP 7: Compute inverse DFT and uncenter
gp = np.real(np.fft.ifft2(Gp)) * centering
print(f"Step 7: Computed inverse DFT and multiplied by (-1)^(x+y)")

# STEP 8: Crop to original size
g = gp[:M, :N]
print(f"Step 8: Cropped result to {M}x{N}")
print(f"\nDone! Lowpass filtered image obtained.")

In [ ]:
# Visualize the complete filtering process
fig, axes = plt.subplots(2, 4, figsize=(20, 10))

# Row 1: Steps 1-4
axes[0, 0].imshow(f, cmap='gray')
axes[0, 0].set_title('Input $f(x,y)$\n$256 \\times 256$', fontsize=11)
axes[0, 0].axis('off')

axes[0, 1].imshow(fp, cmap='gray')
axes[0, 1].set_title('Step 2: Zero-padded\n$512 \\times 512$', fontsize=11)
axes[0, 1].axis('off')

# Show a small region of the centering pattern
axes[0, 2].imshow(fp_centered, cmap='gray')
axes[0, 2].set_title('Step 3: $f_p \\cdot (-1)^{x+y}$', fontsize=11)
axes[0, 2].axis('off')

axes[0, 3].imshow(np.log1p(np.abs(Fp)), cmap='gray')
axes[0, 3].set_title('Step 4: $\\log(1+|F(u,v)|)$\n(centered DFT)', fontsize=11)
axes[0, 3].axis('off')

# Row 2: Steps 5-8
axes[1, 0].imshow(H, cmap='gray')
axes[1, 0].set_title(f'Step 5: $H(u,v)$\nGaussian LP ($D_0={D0}$)', fontsize=11)
axes[1, 0].axis('off')

axes[1, 1].imshow(np.log1p(np.abs(Gp)), cmap='gray')
axes[1, 1].set_title('Step 6: $\\log(1+|G(u,v)|)$\nFiltered spectrum', fontsize=11)
axes[1, 1].axis('off')

axes[1, 2].imshow(gp, cmap='gray')
axes[1, 2].set_title('Step 7: IDFT result\n(full padded size)', fontsize=11)
axes[1, 2].axis('off')

axes[1, 3].imshow(g, cmap='gray')
axes[1, 3].set_title('Step 8: Final $g(x,y)$\n(cropped)', fontsize=11)
axes[1, 3].axis('off')

plt.suptitle('Complete Frequency Domain Lowpass Filtering: All 8 Steps',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Final comparison: original vs filtered at different cutoff frequencies
cutoff_values = [10, 20, 40, 80]

fig, axes = plt.subplots(2, len(cutoff_values) + 1, figsize=(20, 8))

# Show original in the first column
axes[0, 0].imshow(f, cmap='gray')
axes[0, 0].set_title('Original', fontsize=12)
axes[0, 0].axis('off')

axes[1, 0].imshow(np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(f)))), cmap='gray')
axes[1, 0].set_title('Original Spectrum', fontsize=12)
axes[1, 0].axis('off')

for i, D0 in enumerate(cutoff_values):
    # Create Gaussian lowpass filter
    H_lp = np.exp(-(D**2) / (2 * D0**2))
    
    # Apply the complete filtering procedure
    Gp_lp = H_lp * Fp
    gp_lp = np.real(np.fft.ifft2(Gp_lp)) * centering
    g_lp = gp_lp[:M, :N]
    
    axes[0, i+1].imshow(g_lp, cmap='gray')
    axes[0, i+1].set_title(f'$D_0 = {D0}$', fontsize=12)
    axes[0, i+1].axis('off')
    
    axes[1, i+1].imshow(H_lp[P//2-M//2:P//2+M//2, Q//2-N//2:Q//2+N//2], cmap='gray')
    axes[1, i+1].set_title(f'$H(u,v)$, $D_0={D0}$', fontsize=12)
    axes[1, i+1].axis('off')

axes[0, 0].set_ylabel('Filtered Image', fontsize=12)
axes[1, 0].set_ylabel('Filter / Spectrum', fontsize=12)

plt.suptitle('Gaussian Lowpass Filtering at Different Cutoff Frequencies\n'
             'Smaller $D_0$ = More smoothing (removes more high frequencies)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Observations:")
print("  D0 = 10: Very strong smoothing, most details lost, only large structures remain")
print("  D0 = 20: Moderate smoothing, noise reduced, some detail preserved")
print("  D0 = 40: Mild smoothing, noise slightly reduced, most details preserved")
print("  D0 = 80: Minimal smoothing, very similar to original")

In [ ]:
# Cross-section profile comparison
row = M // 2  # middle row

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(f[row, :], 'k-', alpha=0.5, label='Original', linewidth=1)
for D0 in cutoff_values:
    H_lp = np.exp(-(D**2) / (2 * D0**2))
    Gp_lp = H_lp * Fp
    gp_lp = np.real(np.fft.ifft2(Gp_lp)) * centering
    g_lp = gp_lp[:M, :N]
    axes[0].plot(g_lp[row, :], linewidth=2, label=f'$D_0={D0}$')

axes[0].set_title(f'Intensity Profile Along Row {row}', fontsize=12)
axes[0].set_xlabel('Column')
axes[0].set_ylabel('Intensity')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Show the filter profiles (1-D cross-section)
freq_axis = np.arange(Q) - Q // 2
for D0 in cutoff_values:
    H_profile = np.exp(-(freq_axis**2) / (2 * D0**2))
    axes[1].plot(freq_axis, H_profile, linewidth=2, label=f'$D_0={D0}$')

axes[1].set_title('Gaussian Lowpass Filter Profiles', fontsize=12)
axes[1].set_xlabel('Frequency')
axes[1].set_ylabel('$H(u)$')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)
axes[1].set_xlim([-100, 100])

plt.suptitle('Effect of Cutoff Frequency on Smoothing',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Summary

What we learned:

1. **Filtering equation**: $g(x,y) = \mathfrak{F}^{-1}[H(u,v) \cdot F(u,v)]$ -- multiply in frequency domain, then inverse transform

2. **Zero-padding** is essential to avoid wraparound error: pad to $P \geq A+C-1$, $Q \geq B+D-1$

3. **8-step procedure**: pad, center, DFT, construct filter, multiply, IDFT, uncenter, crop

4. **Wraparound error** occurs when circular convolution is mistaken for linear convolution (no padding)

5. **Zero-phase-shift filters** must be real and symmetric so they do not shift spatial features

6. **Spatial kernels and frequency filters** are Fourier transform pairs: wider in space = narrower in frequency

7. **Gaussian lowpass filter** $H(u,v) = e^{-D^2/(2D_0^2)}$ provides smooth, ringing-free lowpass filtering with tunable cutoff $D_0$